In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import regex as re # For String searches
import plotly.graph_objects as go
import plotly.express as px
import time
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
### cell 0 ###

import os
import re

data = pd.read_csv('/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input/speedtest-data-by-ookla/fixed_year_2021_quarter_03.csv')
input_dir = '/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input'
mapping = {'Number of Record': 'Number of Records'}
year_re = re.compile(r'year_(\d+)_quarter')
quarter_re = re.compile(r'quarter_(\d+)\.csv')

mobile_dfs = []
broadband_dfs = []

for root, _, files in os.walk(input_dir):
    for fname in files:
        path = os.path.join(root, fname)
        df = pd.read_csv(path, thousands=',').convert_dtypes()
        # apply any column corrections
        df.rename(columns=mapping, inplace=True)
        # extract year and quarter once per file
        yr = year_re.search(fname).group(1)
        qt = quarter_re.search(fname).group(1)
        df['Year'] = yr
        df['Quarter'] = qt
        # collect by type
        if 'mobile' in fname:
            mobile_dfs.append(df)
        else:
            broadband_dfs.append(df)

# concatenate once
Mobile_df = pd.concat(mobile_dfs, ignore_index=False) if mobile_dfs else pd.DataFrame([], columns=data.columns)
Broadband_df = pd.concat(broadband_dfs, ignore_index=False) if broadband_dfs else pd.DataFrame([], columns=data.columns)

print(Broadband_df.shape)
print(Mobile_df.shape)

# final type casting and sorting
for df in (Mobile_df, Broadband_df):
    df[['Year', 'Quarter']] = df[['Year', 'Quarter']].astype('int64')
    df.sort_values(by=['Year','Quarter'], ascending=True, inplace=True)

factor = 1000
Mobile_df = pd.concat([Mobile_df]*factor*2, ignore_index=True)
Mobile_df.info()

Broadband_df = pd.concat([Broadband_df]*factor, ignore_index=True)
Broadband_df.info()

In [ ]:
### cell 1 ###

unique_countries_broadband = Broadband_df.groupby('Name').count()
unique_countries_broadband.head()

In [ ]:
### cell 2 ###

unique_countries_mobile = Mobile_df.groupby('Name').count()
unique_countries_mobile.head()

In [ ]:
### cell 3 ###

# Check for missing values
Mobile_df.isna().any()

In [ ]:
### cell 4 ###

Broadband_df.isna().any()

In [ ]:
### cell 5 ###

Mobile_df.shape

In [ ]:
### cell 6 ###

unique_countries_mobile[unique_countries_mobile.Year < 10]['Year']

In [ ]:
### cell 7 ###

unique_countries_broadband[unique_countries_broadband.Year < 10]['Year']

In [ ]:
### cell 8 ###

# Compute change statistics for Mobile
Mobile_Stats = (
    Mobile_df
    .groupby('Name')
    .agg(
        first_d=('Avg. Avg D Kbps', 'first'),
        last_d =('Avg. Avg D Kbps', 'last'),
        first_u=('Avg. Avg U Kbps', 'first'),
        last_u =('Avg. Avg U Kbps', 'last'),
        first_l=('Avg Lat Ms',         'first'),
        last_l =('Avg Lat Ms',         'last'),
    )
    .assign(
        Change_Download=lambda df: df['last_d'] - df['first_d'],
        Change_Upload  =lambda df: df['last_u'] - df['first_u'],
        Change_Latency =lambda df: df['last_l'] - df['first_l'],
    )
    [['Change_Download', 'Change_Upload', 'Change_Latency']]
)

# Compute change statistics for Broadband
Broadband_Stats = (
    Broadband_df
    .groupby('Name')
    .agg(
        first_d=('Avg. Avg D Kbps', 'first'),
        last_d =('Avg. Avg D Kbps', 'last'),
        first_u=('Avg. Avg U Kbps', 'first'),
        last_u =('Avg. Avg U Kbps', 'last'),
        first_l=('Avg Lat Ms',         'first'),
        last_l =('Avg Lat Ms',         'last'),
    )
    .assign(
        Change_Download=lambda df: df['last_d'] - df['first_d'],
        Change_Upload  =lambda df: df['last_u'] - df['first_u'],
        Change_Latency =lambda df: df['last_l'] - df['first_l'],
    )
    [['Change_Download', 'Change_Upload', 'Change_Latency']]
)

# Combine into Total_Stats (preserving original column ordering and labels)
Total_Stats = pd.DataFrame({
    'Mobile':    Broadband_Stats['Change_Download'],
    'Broadband': Mobile_Stats['Change_Download'],
})

In [ ]:
### cell 9 ###

ImprovedCountries_B = Broadband_Stats[(Broadband_Stats['Change_Download'] < 3000) &
                                (Broadband_Stats['Change_Download'] > 0)]

In [ ]:
### cell 10 ###

ImprovedCountries_B = Broadband_Stats[(Broadband_Stats['Change_Download'] < 8000) &
                                (Broadband_Stats['Change_Download'] > 3000)]

In [ ]:
### cell 11 ###

# Filter using a single vectorized call instead of two separate comparisons
ImprovedCountries_B = Broadband_Stats.loc[
    Broadband_Stats['Change_Download'].between(8000, 16000, inclusive='neither')
]  # >8000 and <16000

In [ ]:
### cell 12 ###

ImprovedCountries_B = Broadband_Stats[(Broadband_Stats['Change_Download'] < 60000) &
                                (Broadband_Stats['Change_Download'] > 16000)]

In [ ]:
### cell 13 ###

ImprovedCountries_B = Broadband_Stats[(Broadband_Stats['Change_Download'] >= 60000)]

In [ ]:
### cell 14 ###

ImprovedCountries_B = Broadband_Stats[(Broadband_Stats['Change_Download'] < 0)]
Countries = ImprovedCountries_B.index

In [ ]:
### cell 15 ###

Mobile_Stats.sort_values(by=['Change_Download'])

In [ ]:
### cell 16 ###

Broadband_Stats.sort_values(by=['Change_Download'])

In [ ]:
### cell 17 ###

# Define columns to process
cols = ['Avg. Avg D Kbps', 'Avg. Avg U Kbps', 'Avg Lat Ms']

# Mobile
grouped_mobile = Mobile_df.groupby('Name')[cols]
first_mobile = grouped_mobile.first()
last_mobile = grouped_mobile.last()
Mobile_Stats_relative = ((last_mobile - first_mobile) / first_mobile)
Mobile_Stats_relative = Mobile_Stats_relative.rename(columns={
    'Avg. Avg D Kbps': 'Change_Download',
    'Avg. Avg U Kbps': 'Change_Upload',
    'Avg Lat Ms':      'Change_Latency'
})

# Broadband
grouped_broadband = Broadband_df.groupby('Name')[cols]
first_broadband = grouped_broadband.first()
last_broadband = grouped_broadband.last()
Broadband_Stats_relative = ((last_broadband - first_broadband) / first_broadband)
Broadband_Stats_relative = Broadband_Stats_relative.rename(columns={
    'Avg. Avg D Kbps': 'Change_Download',
    'Avg. Avg U Kbps': 'Change_Upload',
    'Avg Lat Ms':      'Change_Latency'
})

In [ ]:
### cell 18 ###

ImprovedCountries_B = Broadband_Stats_relative[(Broadband_Stats_relative['Change_Download'] >= 2)]

In [ ]:
### cell 19 ###

ImprovedCountries_B = Broadband_Stats_relative[(Broadband_Stats_relative['Change_Download'] >= 1) & (Broadband_Stats_relative['Change_Download'] < 2)]

In [ ]:
### cell 20 ###

ImprovedCountries_B = Broadband_Stats_relative[(Broadband_Stats_relative['Change_Download'] >= 0.5) & (Broadband_Stats_relative['Change_Download'] < 1)]

In [ ]:
### cell 21 ###

ImprovedCountries_B = Broadband_Stats_relative[(Broadband_Stats_relative['Change_Download'] >= 0.2) & (Broadband_Stats_relative['Change_Download'] < 0.5)]

In [ ]:
### cell 22 ###

ImprovedCountries_B = Broadband_Stats_relative[(Broadband_Stats_relative['Change_Download'] >= 0) & (Broadband_Stats_relative['Change_Download'] < 0.2)]

In [ ]:
### cell 23 ###

ImprovedCountries_B = Broadband_Stats_relative[(Broadband_Stats_relative['Change_Download'] < 0)]

In [ ]:
### cell 24 ###

Broadband_Stats_relative.sort_values(by=['Change_Download'])

In [ ]:
### cell 25 ###

Mobile_Stats_relative.sort_values(by=['Change_Download'])